<div style="
    width:100%;
    max-width:1080px;
    margin:24px 0 32px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #c2410c 0%, #f59e0b 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h1 style="margin:0 0 14px 0; font-size:34px; line-height:1.2; font-weight:700; color:#ffffff;">Los Estados Ocultos</h1>
<p style="margin:0; color:#cbd5e1; font-size:18px;">El estado oculto es la memoria interna que una red recurrente transporta de un paso al siguiente. En este notebook se presenta como el mecanismo que permite conservar contexto dentro de la secuencia.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Por qué importa</div>
    <div style="color:#f8fafc;">Sin una representación interna del pasado, la red no puede aprovechar dependencias temporales o contextuales.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Qué vas a aprender aquí</div>
    <div style="color:#f8fafc;">A entender qué representa el estado oculto y por qué es clave para el procesamiento secuencial.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Regla práctica que debes recordar</div>
    <div style="color:#f8fafc;">La memoria de una red recurrente no es perfecta; hay que entender qué información puede retener y cuál puede perder.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Contexto mínimo</div>
    <div style="color:#f8fafc;">El estado oculto resume parte de la historia previa de la secuencia en una representación interna actualizable.</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:36px 0 42px 0;">

In [ ]:
# Bloque 1 — Creación de una secuencia sintética
"""Este bloque crea un tensor que representa una secuencia de entrada.
Este bloque importa "torch" para trabajar con tensores en PyTorch.
La variable "secuencia" contiene un tensor que representa una secuencia con 5 pasos temporales, cada uno con una característica de entrada.
Aplicamos el método ".unsqueeze()" a la variable "secuencia" para agregar una dimensión adicional al tensor.
El argumento "0" indica que la nueva dimensión se agregará en la posición 0 del tensor.
La variable "secuencia" contiene valores ordenados que serán procesados paso a paso.
El formato batch-first permite integrarla directamente en una red recurrente."""

import torch

secuencia = torch.tensor([[0.1],
                          [0.3],
                          [0.5],
                          [0.7],
                          [0.9]])

secuencia = secuencia.unsqueeze(0)

secuencia # Muestra el tensor con la dimensión adicional agregada.


In [ ]:
# Bloque 2 — Definición de una RNN simple
"""Este bloque define una red recurrente utilizando nn.RNN.
Este bloque importa "torch.nn" para definir capas de redes neuronales recurrentes en PyTorch.

Aplicamos la clase "nn.RNN()" a la variable "rnn" para definir una red recurrente simple.
El parámetro "input_size=" indica el número de características de entrada en cada paso de la secuencia.
El parámetro "hidden_size=" define el tamaño del estado oculto de la RNN.
El parámetro "num_layers=" especifica el número de capas recurrentes apiladas.
El parámetro "batch_first=" indica si la primera dimensión de la entrada y de la salida representa el lote ("True" en este caso).

La variable "rnn" contiene una instancia de la clase "nn.RNN()", que representa la red recurrente definida.
La red recibe elementos de la secuencia uno a uno y actualiza su estado oculto.
El tamaño del estado oculto influye en la capacidad de la red para representar información temporal."""

import torch.nn as nn

rnn = nn.RNN(input_size=1, hidden_size=4, num_layers=1, batch_first=True)

rnn # Muestra una representación de la red recurrente definida.


In [ ]:
# Bloque 3 — Inicialización del estado oculto
"""Este bloque inicializa el estado oculto con valores cero.
Aplicamos la función "torch.zeros()" a la variable "estado_inicial" para crear un tensor de ceros que representa el estado oculto inicial de la red.
El primer argumento "1" indica el número de capas recurrentes apiladas (en este caso, 1 capa).
El segundo argumento "1" indica el tamaño del lote de entrada (en este caso, una secuencia por lote).
El tercer argumento "4" indica el tamaño del estado oculto de la RNN (el número de unidades en la capa oculta).

La variable "estado_inicial" contiene un tensor de ceros que representa la memoria previa antes de procesar la secuencia.
Este estado se actualizará en cada paso temporal."""

estado_inicial = torch.zeros(1, 1, 4)

estado_inicial # Muestra el estado oculto inicial (tensor de ceros).


In [ ]:
# Bloque 4 — Forward pass y obtención de estados ocultos
"""Este bloque ejecuta el forward pass de la red recurrente.
Asignamos de forma múltiple el resultado de la llamada a la instancia de la clase a las variables "salida" y "estados_ocultos" mediante la expresión "rnn(secuencia, estado_inicial)".
El argumento "secuencia" es el tensor de entrada que contiene los pasos temporales de la secuencia.
El argumento "estado_inicial" es el estado oculto de partida para procesar la secuencia.

La variable "salida" contiene las salidas generadas en cada paso.     
La variable "estados_ocultos" contiene el estado oculto final tras procesar toda la secuencia.
En esta RNN de una capa y una dirección, la última salida coincide con ese estado final."""

salida, estados_ocultos = rnn(secuencia, estado_inicial)

salida, estados_ocultos # Muestra la salida de cada paso y el estado oculto final.


In [ ]:
# Bloque 5 — Inspección de estados ocultos paso a paso
"""Este bloque extrae los estados ocultos generados en cada paso temporal.
Aplicamos los métodos "squeeze()" y "tolist()" a la variable "salida" para obtener la evolución temporal de las salidas de la RNN.
El método "squeeze()" elimina la dimensión del lote, cuyo tamaño es uno.
El argumento "0" indica que se eliminará la primera dimensión del tensor, que corresponde al lote.
El método "tolist()" convierte el tensor resultante en una lista de Python, facilitando su inspección.
La variable "estados_paso" almacena la evolución del estado oculto a lo largo de la secuencia.
La operación permite visualizar cómo la red acumula información progresivamente."""

estados_paso = salida.squeeze(0).tolist()

estados_paso # Muestra la evolución del estado oculto paso a paso.


In [ ]:
# Bloque 6 — Resultados finales
"""Este bloque organiza los resultados del procesamiento recurrente.
La variable "resultado" almacena en forma de diccionario la secuencia original, los estados ocultos paso a paso y el estado final como listas.
La estructura permite analizar cómo la memoria interna evoluciona durante el recorrido secuencial."""

resultado = {
    "secuencia": secuencia.squeeze(0).tolist(),          # El método ".squeeze(0)" elimina la dimensión de lote, de tamaño 1.
    "estados_ocultos_paso_a_paso": estados_paso,
    "estado_final": estados_ocultos.squeeze(0).tolist()  # El método ".squeeze(0)" elimina la dimensión de capa, de tamaño 1.
}

resultado # Muestra el diccionario con la secuencia original, los estados ocultos paso a paso y el estado final.


<div style="
    width:100%;
    max-width:1080px;
    margin:40px 0 24px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #0f766e 0%, #2dd4bf 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h2 style="margin:0 0 14px 0; font-size:30px; line-height:1.2; font-weight:700; color:#ffffff;">Conclusiones</h2>
<p style="margin:0; color:#cbd5e1; font-size:18px;">En este notebook has visto el estado oculto como la memoria operativa que da continuidad a una red recurrente.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Qué conviene recordar</div>
    <div style="color:#f8fafc;">El valor del estado oculto está en resumir contexto útil entre pasos sucesivos.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Error frecuente a evitar</div>
    <div style="color:#f8fafc;">Asumir que la red conserva sin degradación toda la información pasada por el simple hecho de ser recurrente.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente paso natural</div>
    <div style="color:#f8fafc;">Detenerse en las LSTM Gates para dar continuidad directa al hilo del temario.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Conexión con el siguiente concepto</div>
    <div style="color:#f8fafc;">El siguiente notebook se centra en las LSTM Gates y amplía de forma natural lo que acabas de trabajar.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente notebook del temario</div>
    <div style="color:#f8fafc;">10_Redes_recurrentes/04_Las_lstm_gates.ipynb</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:42px 0 24px 0;">